# Assistente de interpretação de exame e geração de laudo descritivo

Este notebook implementa o fluxo:

1. Recebe o texto do resultado de exame em linguagem natural.
2. Usa a API da OpenAI para extrair, em JSON, os quatro parâmetros exigidos pela API FastAPI de `main.py`.
3. Valida os parâmetros localmente.
4. Chama `GET /predict` da API do modelo.
5. Usa a API da OpenAI para produzir um texto descritivo, legível pelo médico, sem inventar diagnóstico.

> **Uso educacional / apoio à decisão:** o conteúdo gerado não substitui interpretação médica, validação do modelo, laudo assinado ou protocolo institucional. Não envie dados identificáveis de pacientes sem uma base legal, consentimento quando aplicável, contrato adequado com o provedor e avaliação de segurança/privacidade. Para testes, use dados anonimizados ou sintéticos.

## 1. Instalação

Execute a célula abaixo uma vez no ambiente virtual do projeto.

In [1]:
%pip install -q openai requests python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


## 2. Configuração

Crie um arquivo `.env` na pasta do notebook:

```env
OPENAI_API_KEY=sua_chave_aqui
OPENAI_MODEL=gpt-4.1-mini
PREDICTION_API_URL=http://127.0.0.1:8000/predict
```

Mantenha `.env` fora do GitHub. Inclua `.env` no `.gitignore`.

In [9]:
import json
import os
from typing import Optional

import requests
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
PREDICTION_API_URL = os.getenv(
    "PREDICTION_API_URL",
    "http://127.0.0.1:8000/predict",
)


if not OPENAI_API_KEY:
    raise RuntimeError("Defina OPENAI_API_KEY no arquivo .env.")

client = OpenAI(api_key=OPENAI_API_KEY)
print("Configuração carregada.")


Configuração carregada.


## 3. Contrato dos parâmetros

O arquivo `main.py` anexado expõe `GET /predict` e espera exatamente:

- `concave_points_mean`
- `concave_points_worst`
- `perimeter_worst`
- `radius_worst`

O extrator não calcula valores ausentes. Se o texto não contiver um parâmetro de forma inequívoca, o fluxo será interrompido antes da chamada ao modelo.

In [10]:
class ParametrosExame(BaseModel):
    concave_points_mean: Optional[float] = Field(default=None)
    concave_points_worst: Optional[float] = Field(default=None)
    perimeter_worst: Optional[float] = Field(default=None)
    radius_worst: Optional[float] = Field(default=None)

    def ausentes(self) -> list[str]:
        return [nome for nome, valor in self.model_dump().items() if valor is None]

    def validar_para_predicao(self) -> dict[str, float]:
        faltantes = self.ausentes()
        if faltantes:
            raise ValueError(
                "Parâmetros ausentes ou ambíguos: " + ", ".join(faltantes)
            )
        valores = self.model_dump()
        for nome, valor in valores.items():
            if not isinstance(valor, (int, float)) or valor < 0:
                raise ValueError(f"Valor inválido para {nome}: {valor}")
        return {nome: float(valor) for nome, valor in valores.items()}

## 4. Converter linguagem natural em parâmetros estruturados

O prompt exige JSON e proíbe inferências. Como segurança adicional, o resultado é validado pelo Pydantic.

In [11]:
def extrair_parametros(texto_exame: str) -> ParametrosExame:
    if not texto_exame or not texto_exame.strip():
        raise ValueError("O texto do exame não pode estar vazio.")

    instrucao = """
Você é um extrator de dados. Não faça diagnóstico e não estime valores.
Extraia somente valores explicitamente informados no texto recebido.
Responda exclusivamente com um objeto JSON válido contendo exatamente:
concave_points_mean, concave_points_worst, perimeter_worst, radius_worst.
Use número decimal com ponto. Se um valor estiver ausente, ilegível ou ambíguo,
use null. Não converta unidades sem que a conversão esteja explicitamente indicada.
""".strip()

    resposta = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": instrucao},
            {"role": "user", "content": texto_exame},
        ],
    )

    conteudo = resposta.choices[0].message.content
    if not conteudo:
        raise RuntimeError("A OpenAI retornou uma resposta vazia.")

    try:
        return ParametrosExame.model_validate(json.loads(conteudo))
    except (json.JSONDecodeError, ValidationError) as erro:
        raise RuntimeError(f"Resposta estruturada inválida: {erro}") from erro

## 5. Chamar a API do modelo

Por padrão, a URL é `http://127.0.0.1:8000/predict`, compatível com o `main.py`. O método é `GET`, conforme o endpoint ativo no arquivo anexado.

In [12]:
""" comentado em 20/10
def chamar_api_predicao(parametros: ParametrosExame) -> dict:
    query_params = parametros.validar_para_predicao()

    try:
        resposta = requests.get(
            PREDICTION_API_URL,
            params=query_params,
            timeout=30,
        )
        resposta.raise_for_status()
    except requests.Timeout as erro:
        raise RuntimeError("A API de predição excedeu o tempo limite.") from erro
    except requests.RequestException as erro:
        raise RuntimeError(f"Falha ao chamar a API de predição: {erro}") from erro

    try:
        dados = resposta.json()
    except ValueError as erro:
        raise RuntimeError("A API de predição não retornou JSON válido.") from erro

    if "output" not in dados:
        raise RuntimeError("A resposta da API não contém o campo 'output'.")

    return dados

"""
import requests

def chamar_api_predicao(parametros):

    payload = {
        "concave_points_mean": parametros.concave_points_mean,
        "concave_points_worst": parametros.concave_points_worst,
        "perimeter_worst": parametros.perimeter_worst,
        "radius_worst": parametros.radius_worst
    }

    response = requests.post(
        "http://localhost:8000/predict",
        json=payload,
        timeout=30
    )

    response.raise_for_status()

    return response.json()

## 6. Gerar laudo descritivo

**Importante:** o `main.py` retorna apenas `{"output": "..."}` e não informa o significado clínico de cada classe. Para evitar uma interpretação errada, configure abaixo o mapa oficial usado no treinamento do modelo. Enquanto ele não for preenchido, o texto apresentará a classe como resultado técnico bruto.

In [13]:
# PREENCHA somente com o mapeamento confirmado no treinamento do modelo.
# Exemplo de estrutura, não de significado clínico:
# MAPA_CLASSES = {"0": "descrição oficial da classe 0", "1": "descrição oficial da classe 1"}
MAPA_CLASSES = {
"0": "Benigno",
"1": "Maligno",
}


def gerar_laudo(
    texto_exame: str,
    parametros: ParametrosExame,
    retorno_api: dict,
) -> str:
    classe_bruta = str(retorno_api["output"])
    significado = MAPA_CLASSES.get(
        classe_bruta,
        "Significado clínico não configurado; apresentar somente como classe técnica bruta.",
    )

    contexto = {
        "texto_de_entrada": texto_exame,
        "parametros_confirmados": parametros.validar_para_predicao(),
        "classe_tecnica_retornada": classe_bruta,
        "significado_oficial_configurado": significado,
    }

    instrucao = """
Redija, em português do Brasil, um relatório descritivo dirigido a um médico.
Use apenas os dados fornecidos no JSON. Não invente fatos, diagnóstico, prognóstico,
tratamento, probabilidade, sensibilidade, especificidade, referência clínica ou
significado de classe. Diferencie claramente valores de entrada e saída do modelo.
Se o significado clínico não estiver configurado, diga explicitamente que a classe
é uma saída técnica que precisa ser interpretada conforme a documentação validada
do modelo. Use linguagem natural, objetiva e profissional, com as seções:
Resumo dos dados analisados; Resultado do modelo; Interpretação e limitações;
Recomendação de validação clínica. Informe que o texto é apoio à decisão e requer
revisão e assinatura de profissional habilitado.
""".strip()

    resposta = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=0.2,
        messages=[
            {"role": "system", "content": instrucao},
            {"role": "user", "content": json.dumps(contexto, ensure_ascii=False)},
        ],
    )

    conteudo = resposta.choices[0].message.content
    if not conteudo:
        raise RuntimeError("A OpenAI retornou um laudo vazio.")
    return conteudo.strip()

## 7. Fluxo completo

A função abaixo executa extração, validação, predição e geração textual. O texto original é enviado à OpenAI, portanto use somente dados devidamente anonimizados e autorizados.

In [14]:
def processar_exame(texto_exame: str) -> dict:
    parametros = extrair_parametros(texto_exame)
    faltantes = parametros.ausentes()

    if faltantes:
        return {
            "status": "revisao_necessaria",
            "mensagem": "Complete os parâmetros antes da predição.",
            "parametros_extraidos": parametros.model_dump(),
            "parametros_ausentes": faltantes,
        }

    retorno_api = chamar_api_predicao(parametros)
    laudo = gerar_laudo(texto_exame, parametros, retorno_api)

    return {
        "status": "sucesso",
        "parametros": parametros.model_dump(),
        "retorno_api": retorno_api,
        "laudo_descritivo": laudo,
    }

## 8. Teste com dados sintéticos

Substitua o exemplo somente por conteúdo anonimizado. Os números abaixo são sintéticos e servem apenas para demonstrar o formato esperado.

In [15]:

texto_exame = """
Dados anonimizados para teste: pontos côncavos médios 0.052; pior valor de
pontos côncavos 0.146; pior perímetro 108.4; pior raio 16.9.
""".strip()

resultado = processar_exame(texto_exame)
#print(json.dumps(resultado, indent=2, ensure_ascii=False))
print(resultado["laudo_descritivo"])

Relatório Descritivo para Avaliação Médica

Resumo dos dados analisados:
Foram fornecidos os seguintes parâmetros quantitativos referentes a características de uma amostra analisada: pontos côncavos médios com valor de 0,052; pior valor de pontos côncavos igual a 0,146; pior perímetro medido em 108,4; e pior raio com medida de 16,9. Estes dados foram confirmados conforme os parâmetros de entrada.

Resultado do modelo:
O modelo computacional retornou a classe técnica "1", cujo significado oficial configurado é "Maligno".

Interpretação e limitações:
A classificação "Maligno" é uma saída técnica do modelo baseada nos parâmetros fornecidos. É importante destacar que esta classificação deve ser interpretada conforme a documentação validada do modelo, considerando que o resultado é uma indicação técnica e não um diagnóstico definitivo. O modelo utiliza os valores de entrada para gerar esta saída, mas não substitui a avaliação clínica detalhada.

Recomendação de validação clínica:
Recomenda-

## 9. Como executar junto com `main.py`

No terminal, na pasta que contém `main.py` e `modelo_pipeline.pkl`:

```powershell
.\venv\Scripts\Activate.ps1
pip install "fastapi[standard]" scikit-learn
fastapi dev main.py
```

Depois, mantenha a API ativa e execute as células do notebook em outro terminal/kernel.

### Antes de uso real

- Confirme a ordem, unidade e pré-processamento das quatro variáveis.
- Documente o significado exato das classes no `MAPA_CLASSES`.
- Valide o modelo em dados adequados à população-alvo.
- Não registre texto clínico, chaves ou identificadores em logs/Git.
- Adicione autenticação, HTTPS, auditoria, controle de acesso e tratamento institucional de dados pessoais/sensíveis.
- Submeta o fluxo à revisão médica, de segurança, privacidade e governança antes de qualquer uso assistencial.